# 🎚 Multi-feature steering — does ablating top-K features control hallucination?

Follow-up to the single-feature steering test (notebook 25). That one found f61723 is predictive (AUROC 0.84) but **not a calibration knob** when ablated alone — generations change in 60% of cases but refusal rate doesn't shift in the predicted direction.

**This notebook** tests whether ablating multiple top entity-recognition features simultaneously controls behavior. If 27B reasoning models route calibration through a circuit (not a single feature), multi-feature ablation should show signal where single-feature didn't.

**Design** — top-K ablation sweep on L11:
1. Re-label 40 entities (20 known + 20 unknown) with v0.0.2 protocol
2. Compute Cohen's d separation for all 65k features at L11, after Pile noise filter
3. Rank top features
4. For each K ∈ {0, 5, 20, 50, 200}, ablate top-K simultaneously during generation
5. Score refusal rate per condition

**Verdict thresholds** (effect on unknown, ablate direction expected to ↓ refusal):
- Δ ≤ -15pp: CAUSAL · multi-feature controls calibration
- -5 to -15pp: PARTIAL · small but real effect
- |Δ| < 5pp: NULL · ablation doesn't move refusal
- Δ ≥ +5pp: INVERTED · wrong-direction effect (epiphenomenal pathway)

**Cost**: ~$10 GPU + ~40 min on RTX 6000 Pro. Self-contained — runs from a fresh Colab kernel.

Reference: [single-feature steering result](https://huggingface.co/caiovicentino1/qwen36-27b-sae-papergrade/blob/main/steering_v0_0_1.json) · [predecessor v0.0.2 AUROC 0.84](https://huggingface.co/caiovicentino1/qwen36-27b-sae-papergrade/blob/main/hallucination_v0_0_2.json).

In [ ]:
!pip install -q -U transformers accelerate safetensors huggingface_hub datasets matplotlib tqdm requests
import torch, transformers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)

## 1. Config + load model + L11 SAE

In [ ]:
HF_SAE_REPO   = 'caiovicentino1/qwen36-27b-sae-papergrade'
HF_BASE_MODEL = 'Qwen/Qwen3.6-27B'
STEER_LAYER   = 11
D_MODEL       = 5120
D_SAE         = 65_536
K             = 128

K_SWEEP = [0, 5, 20, 50, 200]   # 0 = baseline (no ablation)

N_KNOWN_EVAL   = 20
N_UNKNOWN_EVAL = 20
PER_TYPE_CANDIDATES = 80
MAX_GEN_TOKENS = 80
PILE_THRESHOLD = 0.02
N_PILE_TOKENS = 2000

import os, math, json, time, random, re
import numpy as np
import requests
from collections import Counter
from datetime import datetime, timezone
random.seed(0); torch.manual_seed(0); np.random.seed(0)

from huggingface_hub import login, hf_hub_download, HfApi
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

from transformers import AutoTokenizer, AutoModelForImageTextToText
from safetensors.torch import load_file
import torch.nn.functional as F

device = 'cuda'
tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    HF_BASE_MODEL, dtype=torch.bfloat16, attn_implementation='sdpa',
    device_map='cuda', trust_remote_code=True,
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

class TopKSAE(torch.nn.Module):
    def __init__(self, sd, k):
        super().__init__()
        self.W_enc = torch.nn.Parameter(sd['W_enc'].to(torch.bfloat16), requires_grad=False)
        self.b_enc = torch.nn.Parameter(sd['b_enc'].to(torch.bfloat16), requires_grad=False)
        self.W_dec = torch.nn.Parameter(sd['W_dec'].to(torch.bfloat16), requires_grad=False)
        self.b_dec = torch.nn.Parameter(sd['b_dec'].to(torch.bfloat16), requires_grad=False)
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        vals = F.relu(vals)
        z = torch.zeros_like(pre)
        z.scatter_(-1, idx, vals)
        return z
    def decode(self, z):
        return z @ self.W_dec + self.b_dec

sae = TopKSAE(load_file(hf_hub_download(HF_SAE_REPO, f'sae_L{STEER_LAYER}_latest.safetensors')), K).to(device).eval()
layer_mod = model.model.language_model.layers[STEER_LAYER]
print(f'  ✓ SAE L{STEER_LAYER} loaded · vram free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## 2. Pull entities + label 40 (matches v0.0.2 protocol)

In [ ]:
from tqdm.auto import tqdm

FERRANDO_BASE = 'https://raw.githubusercontent.com/javiferran/sae_entities/main/dataset/processed/entities'
ENTITY_TYPES = ['player', 'movie']
raw = {t: requests.get(f'{FERRANDO_BASE}/{t}.json', timeout=60).json() for t in ENTITY_TYPES}
rng = random.Random(0)
candidates = []
for t in ENTITY_TYPES:
    sample = rng.sample(raw[t], min(PER_TYPE_CANDIDATES, len(raw[t])))
    for ent in sample:
        candidates.append({'type': t, **ent})
print(f'  candidates to label: {len(candidates)}')

ATTRS = {
    'player': {
        'place_birth':  "What is the place of birth of the basketball player '{entity}'? Answer in just one or two words.",
        'date_birth':   "In what year was the basketball player '{entity}' born? Answer with just a year.",
        'teams_list':   "What was a team that the basketball player '{entity}' played for? Answer with just the team name.",
    },
    'movie': {
        'directors':    "Who directed the movie '{entity}'? Answer with just the director's name.",
        'release_year': "In what year was the movie '{entity}' released? Answer with just a year.",
        'genres':       "What is one genre of the movie '{entity}'? Answer with just one word.",
    },
}

REFUSAL_RE = re.compile('|'.join([
    r"i (?:don'?t|do not) (?:know|have)",
    r"i'?m (?:sorry|not sure|not familiar|unable)",
    r"i (?:cannot|can'?t) (?:provide|verify|confirm|find)",
    r"there (?:is|seems to be) (?:no|insufficient|limited) (?:information|data|record)",
    r"unable to (?:find|locate|verify)",
    r"(?:no|not enough|insufficient) (?:public(?:ly available)?\s+)?(?:information|data|record)",
    r"i don't have (?:specific|enough|reliable) (?:information|details|data)",
    r"there'?s no (?:widely|publicly|reliable) (?:known|available)",
    r"no widely recognized (?:public figure|professional|celebrity|historical figure)",
]), re.IGNORECASE)

def is_refusal(t): return bool(REFUSAL_RE.search(t or ''))
def normalise(s): return re.sub(r'[^a-z0-9]+', ' ', (s or '').lower()).strip()
def attr_match(ans, gt):
    if isinstance(gt, list): return any(attr_match(ans, x) for x in gt)
    if not gt or not ans: return False
    a, g = normalise(str(ans)), normalise(str(gt))
    return bool(g) and (g in a or a in g)

def chat_short(q):
    msgs = [
        {'role': 'system', 'content': 'Answer concisely and directly. Do not show reasoning. If you do not know, say "I do not know".'},
        {'role': 'user', 'content': q},
    ]
    try:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(ids['input_ids'], attention_mask=ids['attention_mask'],
                              max_new_tokens=80, do_sample=False, pad_token_id=tok.eos_token_id)
    ans = tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    if 'thinking process' in ans.lower():
        parts = [p for p in ans.split('\n') if p.strip()]
        ans = parts[-1] if parts else ans
    return ans

labelled = []
for ent in tqdm(candidates, desc='label'):
    name, type_ = ent['entity'], ent['type']
    gt = {}
    for a in ent.get('attributes', []):
        gt.setdefault(a['attribute_type'], []).append(a['attribute_value'])
    avail = [k for k in ATTRS[type_] if k in gt]
    if len(avail) < 2: continue
    selected = avail[:3]
    nc, nr = 0, 0
    for at in selected:
        a = chat_short(ATTRS[type_][at].format(entity=name))
        if is_refusal(a): nr += 1
        elif attr_match(a, gt[at]): nc += 1
    if nc >= 2 and nr == 0: cls = 'known'
    elif nc == 0 and nr >= 1: cls = 'unknown'
    else: cls = 'middle'
    labelled.append({'type': type_, 'entity': name, 'class': cls})
    c = Counter(l['class'] for l in labelled)
    if c.get('known', 0) >= N_KNOWN_EVAL and c.get('unknown', 0) >= N_UNKNOWN_EVAL:
        break

known_eval   = [l for l in labelled if l['class'] == 'known'][:N_KNOWN_EVAL]
unknown_eval = [l for l in labelled if l['class'] == 'unknown'][:N_UNKNOWN_EVAL]
all_eval = [(e, 'known') for e in known_eval] + [(e, 'unknown') for e in unknown_eval]
print(f'\neval set: {len(known_eval)} known + {len(unknown_eval)} unknown')
print(f'  known sample: {[e["entity"] for e in known_eval[:3]]}')
print(f'  unknown sample: {[e["entity"] for e in unknown_eval[:3]]}')

## 3. Capture activations + Pile filter + rank top features

In [ ]:
from datasets import load_dataset

PROMPT_TPL = "What can you tell me about '{entity}'?"

_cap = {}
def cap_hook(mod, inp, out):
    _cap['h'] = out[0] if isinstance(out, tuple) else out
    return out

def find_pos(prompt):
    ids_ = tok(prompt, return_tensors='pt')['input_ids'][0].tolist()
    cq = tok.encode("'?", add_special_tokens=False)
    for i in range(len(ids_)-len(cq), -1, -1):
        if ids_[i:i+len(cq)] == cq: return i-1
    return len(ids_)-2

def encode_set(entries, label):
    zs = []
    h = layer_mod.register_forward_hook(cap_hook)
    with torch.no_grad():
        for ent in tqdm(entries, desc=label):
            prompt = PROMPT_TPL.format(entity=ent['entity'])
            ids = tok(prompt, return_tensors='pt')['input_ids'].to(device)
            pos = find_pos(prompt)
            _ = model(ids)
            resid = _cap['h'][0, pos].to(torch.bfloat16)
            z = sae.encode(resid.unsqueeze(0))[0].float().cpu().numpy()
            zs.append(z)
    h.remove()
    return np.stack(zs, axis=0)

Z_k = encode_set(known_eval, 'capture known')
Z_u = encode_set(unknown_eval, 'capture unknown')

# Pile filter
pile_ds = load_dataset('NeelNanda/pile-10k', split='train', streaming=True)
pt = []
for x in pile_ds:
    pt.append(x['text'])
    if len(pt) >= 50: break
pile_ids = tok(' '.join(pt), return_tensors='pt', max_length=N_PILE_TOKENS, truncation=True)['input_ids'].to(device)
h = layer_mod.register_forward_hook(cap_hook)
with torch.no_grad(): _ = model(pile_ids)
h.remove()
pile_z = sae.encode(_cap['h'][0].to(torch.bfloat16))
pile_fr = (pile_z > 0).float().mean(0).cpu().numpy()
print(f'\nPile: {(pile_fr > PILE_THRESHOLD).sum()} features dropped (>{PILE_THRESHOLD*100:.1f}% rate)')

# Cohen's d separation, ranked
mu_k, mu_u = Z_k.mean(0), Z_u.mean(0)
sd = np.sqrt((Z_k.std(0)**2 + Z_u.std(0)**2) / 2) + 1e-9
sep = (mu_k - mu_u) / sd
sep[pile_fr > PILE_THRESHOLD] = 0
top_feats_all = np.argsort(np.abs(sep))[::-1].tolist()

print('\nTop 10 features by |Cohen\'s d| (Pile-filtered):')
for r in range(10):
    f = top_feats_all[r]
    print(f'  rank {r+1:2d}: f{f:>5d}  sep={sep[f]:+.3f}  μ_k={mu_k[f]:.2f} μ_u={mu_u[f]:.2f}')

## 4. Multi-feature ablation hook + K-sweep generation

In [ ]:
_K_active = {'feats': []}
def multi_ablate_hook(mod, inp, out):
    if not _K_active['feats']:
        return out
    h = out[0] if isinstance(out, tuple) else out
    orig_dtype = h.dtype
    flat = h.reshape(-1, D_MODEL).to(torch.bfloat16)
    z = sae.encode(flat)
    recon_orig = sae.decode(z).to(torch.float32)
    err = flat.to(torch.float32) - recon_orig
    z_mod = z.clone()
    for f in _K_active['feats']:
        z_mod[:, f] = 0
    recon_new = sae.decode(z_mod).to(torch.float32)
    new_h = (recon_new + err).to(orig_dtype).reshape(h.shape)
    if isinstance(out, tuple):
        return (new_h,) + out[1:]
    return new_h

_steer = layer_mod.register_forward_hook(multi_ablate_hook)

def gen_K(K, entity_name):
    _K_active['feats'] = top_feats_all[:K] if K > 0 else []
    msgs = [{'role': 'user', 'content': f"What can you tell me about {entity_name}? Be specific."}]
    try:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(ids['input_ids'], attention_mask=ids['attention_mask'],
                              max_new_tokens=MAX_GEN_TOKENS, do_sample=False, pad_token_id=tok.eos_token_id)
    _K_active['feats'] = []
    return tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# Smoke test on 1 known + 1 unknown × all K values
print('Smoke test:')
for label, ent in [('known', known_eval[0]), ('unknown', unknown_eval[0])]:
    print(f"\n[{label}] {ent['entity']}")
    for k in K_SWEEP:
        r = gen_K(k, ent['entity'])
        print(f'  K={k:>3d} | refusal={is_refusal(r)} | {r[:140]}')

## 5. Run full K sweep — 40 entities × 5 conditions = 200 generations

In [ ]:
results = []
for ent, ent_class in tqdm(all_eval, desc='K sweep'):
    row = {'entity': ent['entity'], 'type': ent['type'], 'class': ent_class}
    for k in K_SWEEP:
        txt = gen_K(k, ent['entity'])
        row[f'K{k}_text'] = txt
        row[f'K{k}_refusal'] = is_refusal(txt)
    results.append(row)

print(f'\ngenerated {len(results) * len(K_SWEEP)} responses')

## 6. Effect sizes + verdict

In [ ]:
def rate(rows, k):
    return sum(r[f'K{k}_refusal'] for r in rows) / max(len(rows), 1)

mk = [r for r in results if r['class'] == 'known']
mu = [r for r in results if r['class'] == 'unknown']

print('═' * 64)
print('Multi-feature ablation @ L11 — refusal rate by K')
print('═' * 64)
print('              ' + '   '.join(f'K={k:>3d}' for k in K_SWEEP))
print(f'  KNOWN   n={len(mk):2d}  ' + '  '.join(f'{rate(mk,k):>5.1%}' for k in K_SWEEP))
print(f'  UNKNOWN n={len(mu):2d}  ' + '  '.join(f'{rate(mu,k):>5.1%}' for k in K_SWEEP))

delta_unknown_topK = rate(mu, K_SWEEP[-1]) - rate(mu, 0)
delta_unknown_K50  = rate(mu, 50) - rate(mu, 0)
delta_unknown_K20  = rate(mu, 20) - rate(mu, 0)
delta_unknown_K5   = rate(mu, 5) - rate(mu, 0)

print(f'\nΔ refusal on UNKNOWN (vs K=0 baseline):')
print(f'  K=5:   {delta_unknown_K5:+.1%}')
print(f'  K=20:  {delta_unknown_K20:+.1%}')
print(f'  K=50:  {delta_unknown_K50:+.1%}')
print(f'  K={K_SWEEP[-1]}: {delta_unknown_topK:+.1%}')
print(f'  expected: NEGATIVE (ablating top entity-recognition features → model treats as known → less refusal)')

# Same for known
delta_known_topK = rate(mk, K_SWEEP[-1]) - rate(mk, 0)
print(f'\nΔ refusal on KNOWN (vs K=0 baseline):')
print(f'  K={K_SWEEP[-1]}: {delta_known_topK:+.1%}')

# Text change rates — how often does top-K alter the generation?
for k in K_SWEEP[1:]:
    nk = sum(1 for r in mk if r[f'K{k}_text'] != r['K0_text'])
    nu = sum(1 for r in mu if r[f'K{k}_text'] != r['K0_text'])
    print(f'  K={k:>3d}: known {nk}/{len(mk)} texts changed, unknown {nu}/{len(mu)} texts changed')

# Verdict
if delta_unknown_topK <= -0.15:
    verdict, color = 'CAUSAL · multi-feature ablation controls calibration', '\u2705'
elif delta_unknown_topK <= -0.05:
    verdict, color = 'PARTIAL · small but real effect in expected direction', '\u26a0\ufe0f'
elif abs(delta_unknown_topK) < 0.05:
    verdict, color = 'NULL · multi-feature ablation does not move refusal', '\u26a0\ufe0f'
else:
    verdict, color = 'INVERTED · ablation moves refusal in unexpected direction', '\u274c'

print(f'\n{color}  VERDICT: {verdict}')
print(f'   max effect (Δ on unknown @ K={K_SWEEP[-1]}): {delta_unknown_topK:+.1%}')

## 7. Qualitative inspection — text shifts across K

In [ ]:
# 3 known + 3 unknown showing how text changed
print('═' * 64)
print('KNOWN entities — text across K=0, 50, 200')
print('═' * 64)
for r in mk[:3]:
    print(f"\n● {r['entity']}")
    for k in [0, 50, 200]:
        if k in K_SWEEP:
            print(f'  K={k:>3d}: {r[f"K{k}_text"][:180]}')

print('\n' + '═' * 64)
print('UNKNOWN entities — text across K=0, 50, 200')
print('═' * 64)
for r in mu[:3]:
    print(f"\n● {r['entity']}")
    for k in [0, 50, 200]:
        if k in K_SWEEP:
            print(f'  K={k:>3d}: {r[f"K{k}_text"][:180]}')

## 8. Save artifact to HF

In [ ]:
out = {
    'version':       'v0.0.1',
    'experiment':    f'multi_feature_top_K_ablation_L{STEER_LAYER}',
    'predecessor_single_feature_steering': 'steering_v0_0_1.json (NOT_CAUSAL_FOR_CALIBRATION on f61723 alone)',
    'predecessor_correlational_auroc':     0.8379,
    'K_sweep':       K_SWEEP,
    'pile_threshold': PILE_THRESHOLD,
    'top_5_features': [int(top_feats_all[r]) for r in range(5)],
    'separation_at_top_5': [float(sep[top_feats_all[r]]) for r in range(5)],
    'rates_known':   {f'K{k}': rate(mk, k) for k in K_SWEEP},
    'rates_unknown': {f'K{k}': rate(mu, k) for k in K_SWEEP},
    'delta_unknown_vs_baseline': {f'K{k}': rate(mu, k) - rate(mu, 0) for k in K_SWEEP},
    'delta_known_vs_baseline':   {f'K{k}': rate(mk, k) - rate(mk, 0) for k in K_SWEEP},
    'text_changed_known':        {f'K{k}': sum(1 for r in mk if r[f'K{k}_text'] != r['K0_text']) for k in K_SWEEP[1:]},
    'text_changed_unknown':      {f'K{k}': sum(1 for r in mu if r[f'K{k}_text'] != r['K0_text']) for k in K_SWEEP[1:]},
    'n_known':       len(mk),
    'n_unknown':     len(mu),
    'verdict':       verdict,
    'timestamp':     datetime.now(timezone.utc).isoformat(),
}
with open('/tmp/multi_steering.json', 'w') as f:
    json.dump(out, f, indent=2)

api = HfApi()
api.upload_file(
    path_or_fileobj='/tmp/multi_steering.json',
    path_in_repo='multi_feature_steering_v0_0_1.json',
    repo_id=HF_SAE_REPO,
    commit_message=f'Multi-feature ablation v0.0.1 — {verdict.split(chr(183))[0].strip()} (Δunk@K{K_SWEEP[-1]}={delta_unknown_topK:+.1%})',
)
print(f'\n✓ uploaded to https://huggingface.co/{HF_SAE_REPO}/blob/main/multi_feature_steering_v0_0_1.json')
print(f'verdict: {verdict}')